# Missing-family T4 allocator-residency campaign
Scale scan workloads first, then threshold allocator capacity. No production model is fitted unless the strict gate is met.

In [ ]:
import json,sys,importlib,zipfile,collections,statistics
from pathlib import Path
REPO=Path('/content/jaxoom')
PINNED_COMMIT='573b3c748608bf31a3e940d2eb73305ed89821a4'

## Install pinned infrastructure


In [ ]:
!pip install -q "jax[cuda12]==0.11.0"
!rm -rf /content/jaxoom
!git clone -q https://github.com/Slavov88/jaxoom.git /content/jaxoom
!cd /content/jaxoom && git checkout -q 573b3c748608bf31a3e940d2eb73305ed89821a4 && pip install -q -e .


## Verify T4 and validate scan tracing


In [ ]:
sys.path.insert(0,'/content/jaxoom/src'); sys.path.insert(0,'/content/jaxoom/experiments'); sys.modules.pop('jaxoom',None); importlib.invalidate_caches()
import jax,jaxlib,jaxoom
assert jax.__version__=='0.11.0' and jaxlib.__version__=='0.11.0' and jax.default_backend()=='gpu'
d=jax.devices()[0]; assert 'T4' in getattr(d,'device_kind',str(d)),d
from runtime_validation import workload_from_config
scan_rows=[]
for family,config in [('mlp_scan',{'batch':2,'width':8,'layers':3,'retain':True}),('training_scan',{'batch':2,'width':8,'layers':3}),('autodiff_scan',{'batch':2,'width':8,'layers':3})]:
 w=workload_from_config(family,config,'float32'); j=jax.make_jaxpr(w.fn)(*w.abstract_args).jaxpr; r=jaxoom.estimate(w.fn,*w.abstract_args); scan_rows.append({'family':family,'equation_count':len(j.eqns),'scan_count':sum(str(e.primitive)=='scan' for e in j.eqns),'structural_peak_bytes':r.estimated_peak_bytes})
assert all(r['scan_count']==1 for r in scan_rows)
Path('/content/environment.json').write_text(json.dumps({'device':str(d),'device_kind':getattr(d,'device_kind',None),'jax_version':jax.__version__,'jaxlib_version':jaxlib.__version__,'backend':jax.default_backend(),'pinned_commit':PINNED_COMMIT,'preallocate':False},indent=2)+'\n')
Path('/content/scan_validation.json').write_text(json.dumps({'status':'OBSERVED','rows':scan_rows},indent=2)+'\n')
print(scan_rows)


## Calibrate and search workload scale


In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --calibrate --fractions 0.04,0.05,0.06,0.07,0.08,0.09,0.10,0.12,0.15,0.20,0.25,0.30,0.35 --output /content/fraction_map.json --timeout 30
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_missing_family_campaign.py --screening /content/scan_screening.json --screen-only
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_missing_family_campaign.py --screening /content/scan_screening.json --scale-output /content/scale_searches.json --fraction 0.06 --timeout 60 --max-probes 100 --max-per-family 2


## Threshold selected scale boundaries and convolution control


In [ ]:
sel=json.loads(Path('/content/scale_searches.json.manifest.json').read_text())['selected']
conv=json.loads((REPO/'experiments/allocator_residency_t4_long_new_thresholds.json').read_text()).get('rows',[])
sel += [{'configuration_id':r['configuration_id'],'family':r['family'],'configuration':r['configuration'],'dtype':r['dtype']} for r in conv if r.get('family')=='convolution']
u={r['configuration_id']:r for r in sel}; Path('/content/candidate_configurations.json').write_text(json.dumps({'status':'MANIFEST','candidates':list(u.values())},indent=2)+'\n')
print(len(u),collections.Counter(r['family'] for r in u.values()))
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --manifest /content/candidate_configurations.json --fraction-map /content/fraction_map.json --output /content/raw_probes.json --timeout 60 --max-probes 100


## Derive distinct thresholds and package


In [ ]:
def out(r): return (r.get('execution_statuses') or [r.get('outcome') or r.get('status') or r.get('compile_status') or 'OTHER_FAILURE'])[0]
def cap(r): return ((r.get('snapshots') or [{}])[0]).get('allocator_limit_bytes')
def derive(rows):
 g={}
 for r in rows:g.setdefault(r.get('configuration_id'),[]).append(r)
 a=[]
 for cid,rs in g.items():
  f=[r for r in rs if out(r)=='FIT' and cap(r) is not None]; o=[r for r in rs if out(r) in {'COMPILE_OOM','EXECUTION_OOM'} and cap(r) is not None]
  if f and o:
   lo=max(cap(r) for r in o); hi=min(cap(r) for r in f); a.append({'configuration_id':cid,'family':rs[0].get('family'),'configuration':rs[0].get('configuration'),'dtype':rs[0].get('dtype'),'known_oom_capacity':lo,'known_fit_capacity':hi,'bracket_width_bytes':hi-lo})
 return a
raw=json.loads(Path('/content/raw_probes.json').read_text()).get('rows',[]); new=derive(raw)
old=json.loads((REPO/'experiments/allocator_residency_t4_refined_thresholds_2026-09-09.json').read_text())['rows']; phase=json.loads((REPO/'experiments/allocator_residency_t4_phase_c_new_thresholds.json').read_text())['rows']; prior=json.loads((REPO/'experiments/allocator_residency_t4_long_new_thresholds.json').read_text()).get('rows',[])
def key(r): return (r.get('family'),json.dumps(r.get('configuration'),sort_keys=True),r.get('dtype'))
allr=list({key(r):r for r in old+phase+prior+new}.values()); fam=collections.Counter(r['family'] for r in allr); dt=collections.Counter(r['dtype'] for r in allr); missing=sum(fam.get(x,0) for x in ('mlp_scan','training_scan','autodiff_scan'))
gate=len(allr)>=15 and len(fam)>=5 and missing>=4 and statistics.median(r['bracket_width_bytes'] for r in allr)<=64*1024**2
summary={'status':'OBSERVED','new_thresholds':len(new),'distinct_thresholds':len(allr),'families':dict(fam),'dtypes':dict(dt),'outcomes':dict(collections.Counter(out(r) for r in raw)),'model_gate':{'met':gate,'missing_family_thresholds':missing}}
Path('/content/new_thresholds.json').write_text(json.dumps({'status':'OBSERVED','rows':new},indent=2)+'\n'); Path('/content/all_distinct_thresholds.json').write_text(json.dumps({'status':'OBSERVED','rows':allr},indent=2)+'\n'); Path('/content/threshold_summary.json').write_text(json.dumps(summary,indent=2)+'\n'); Path('/content/campaign_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
reason='modeling_gate_not_met' if not gate else 'requires_offline_modeling_pass'
for name in ('model_split.json','model_comparison.json','family_holdout.json'): Path('/content/'+name).write_text(json.dumps({'status':'NOT_RUN','reason':reason},indent=2)+'\n')
Path('/content/paired_rtx_t4.json').write_text(json.dumps({'status':'OBSERVED','rows':[]},indent=2)+'\n'); print(json.dumps(summary,indent=2))


## Package results


In [ ]:
files=['environment.json','fraction_map.json','scan_validation.json','scale_searches.json','candidate_configurations.json','raw_probes.json','new_thresholds.json','all_distinct_thresholds.json','threshold_summary.json','model_split.json','model_comparison.json','family_holdout.json','paired_rtx_t4.json','campaign_summary.json']
with zipfile.ZipFile('/content/jaxoom_t4_missing_family_residency.zip','w',zipfile.ZIP_DEFLATED) as z:
 for name in files:
  p=Path('/content')/name
  if p.exists(): z.write(p,arcname=name)
from google.colab import files
files.download('/content/jaxoom_t4_missing_family_residency.zip')
